# Top Selling Products Analysis

This notebook analyzes transaction data to find top selling products using both DataFrame API and RDD API.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, desc

spark = SparkSession.builder \
    .appName('Top Selling Products') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Data Format

The transaction file (`txn.txt`) contains:
- `TransactionNo`: Unique transaction identifier
- `Date`: Transaction date
- `ProductNo`: Product identifier
- `ProductName`: Name of the product
- `Price`: Price per unit
- `Quantity`: Quantity purchased
- `CustomerNo`: Customer identifier
- `Country`: Customer's country

## Method 1: DataFrame API

In [ ]:
# Load transaction data
txn_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", "\t") \
    .load("/content/sample_data/txn.txt")

print("Schema:")
txn_df.printSchema()

print("\nSample data:")
txn_df.show(10, truncate=False)

In [ ]:
# Basic statistics
print(f"Total transactions: {txn_df.count()}")
print(f"Unique products: {txn_df.select('ProductNo').distinct().count()}")
print(f"Unique customers: {txn_df.select('CustomerNo').distinct().count()}")
print(f"Countries: {txn_df.select('Country').distinct().count()}")

### Calculate Total Sales per Product

In [ ]:
# Add TotalSale column (Price * Quantity)
txn_with_total = txn_df.withColumn(
    "TotalSale", 
    col("Price") * col("Quantity")
)

print("Data with TotalSale column:")
txn_with_total.select(
    "TransactionNo", "ProductName", "Price", "Quantity", "TotalSale"
).show(10)

### Top 10 Products by Quantity Sold

In [ ]:
# Group by product and sum quantity
top_by_quantity = txn_df \
    .groupBy("ProductNo", "ProductName") \
    .agg(spark_sum("Quantity").alias("TotalQuantity")) \
    .orderBy(desc("TotalQuantity")) \
    .limit(10)

print("Top 10 Products by Quantity Sold:")
top_by_quantity.show(truncate=False)

### Top 10 Products by Revenue

In [ ]:
# Group by product and sum total sales
top_by_revenue = txn_with_total \
    .groupBy("ProductNo", "ProductName") \
    .agg(spark_sum("TotalSale").alias("TotalRevenue")) \
    .orderBy(desc("TotalRevenue")) \
    .limit(10)

print("Top 10 Products by Revenue:")
top_by_revenue.show(truncate=False)

### Top 10 Products with Complete Statistics

In [ ]:
# Complete product statistics
product_stats = txn_with_total \
    .groupBy("ProductNo", "ProductName") \
    .agg(
        spark_sum("Quantity").alias("TotalQuantity"),
        spark_sum("TotalSale").alias("TotalRevenue"),
        F.count("TransactionNo").alias("NumberOfTransactions"),
        F.avg("Price").alias("AvgPrice")
    ) \
    .orderBy(desc("TotalRevenue")) \
    .limit(10)

print("Top 10 Products with Complete Statistics:")
product_stats.show(truncate=False)

## Method 2: Spark SQL

In [ ]:
# Register DataFrame as temp view
txn_df.createOrReplaceTempView("transactions")

In [ ]:
# Top 10 by quantity using SQL
top_quantity_sql = spark.sql("""
    SELECT 
        ProductNo,
        ProductName,
        SUM(Quantity) as TotalQuantity
    FROM transactions
    GROUP BY ProductNo, ProductName
    ORDER BY TotalQuantity DESC
    LIMIT 10
""")

print("Top 10 Products by Quantity (SQL):")
top_quantity_sql.show(truncate=False)

In [ ]:
# Top 10 by revenue using SQL
top_revenue_sql = spark.sql("""
    SELECT 
        ProductNo,
        ProductName,
        SUM(Price * Quantity) as TotalRevenue,
        SUM(Quantity) as TotalQuantity,
        COUNT(*) as NumTransactions,
        ROUND(AVG(Price), 2) as AvgPrice
    FROM transactions
    GROUP BY ProductNo, ProductName
    ORDER BY TotalRevenue DESC
    LIMIT 10
""")

print("Top 10 Products by Revenue (SQL):")
top_revenue_sql.show(truncate=False)

## Method 3: RDD API

In [ ]:
# Load as RDD
txn_rdd = spark.sparkContext.textFile("/content/sample_data/txn.txt")

# Remove header
header = txn_rdd.first()
txn_data = txn_rdd.filter(lambda line: line != header)

print("Sample RDD data (first 5 lines):")
for line in txn_data.take(5):
    print(line)

### Parse RDD Data

In [ ]:
# Parse lines and extract fields
def parse_line(line):
    fields = line.split('\t')
    try:
        return {
            'transaction_no': fields[0],
            'date': fields[1],
            'product_no': fields[2],
            'product_name': fields[3],
            'price': float(fields[4]),
            'quantity': int(fields[5]),
            'customer_no': fields[6],
            'country': fields[7]
        }
    except (ValueError, IndexError):
        return None

# Parse and filter out invalid lines
parsed_rdd = txn_data.map(parse_line).filter(lambda x: x is not None)

print("Parsed data sample:")
for record in parsed_rdd.take(3):
    print(record)

### Top 10 Products by Quantity (RDD)

In [ ]:
# Map to (product_name, quantity) and reduce by key
product_quantity = parsed_rdd \
    .map(lambda x: (x['product_name'], x['quantity'])) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(10)

print("Top 10 Products by Quantity (RDD):")
print(f"{'Product Name':<50} {'Total Quantity':>15}")
print("="*67)
for product, quantity in product_quantity:
    print(f"{product:<50} {quantity:>15,}")

### Top 10 Products by Revenue (RDD)

In [ ]:
# Map to (product_name, revenue) and reduce by key
product_revenue = parsed_rdd \
    .map(lambda x: (x['product_name'], x['price'] * x['quantity'])) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(10)

print("Top 10 Products by Revenue (RDD):")
print(f"{'Product Name':<50} {'Total Revenue':>15}")
print("="*67)
for product, revenue in product_revenue:
    print(f"{product:<50} ${revenue:>14,.2f}")

### Complete Product Statistics (RDD with aggregateByKey)

In [ ]:
# Calculate comprehensive statistics using aggregateByKey
def create_combiner(record):
    return {
        'quantity': record['quantity'],
        'revenue': record['price'] * record['quantity'],
        'count': 1,
        'price_sum': record['price']
    }

def merge_value(acc, record):
    acc['quantity'] += record['quantity']
    acc['revenue'] += record['price'] * record['quantity']
    acc['count'] += 1
    acc['price_sum'] += record['price']
    return acc

def merge_combiners(acc1, acc2):
    return {
        'quantity': acc1['quantity'] + acc2['quantity'],
        'revenue': acc1['revenue'] + acc2['revenue'],
        'count': acc1['count'] + acc2['count'],
        'price_sum': acc1['price_sum'] + acc2['price_sum']
    }

# Apply aggregateByKey
product_stats_rdd = parsed_rdd \
    .map(lambda x: (x['product_name'], x)) \
    .aggregateByKey(
        None,
        lambda acc, v: create_combiner(v) if acc is None else merge_value(acc, v),
        merge_combiners
    ) \
    .sortBy(lambda x: x[1]['revenue'], ascending=False) \
    .take(10)

print("Top 10 Products - Complete Statistics (RDD):")
print(f"{'Product':<40} {'Qty':>10} {'Revenue':>15} {'Txns':>8} {'Avg Price':>12}")
print("="*90)
for product, stats in product_stats_rdd:
    avg_price = stats['price_sum'] / stats['count']
    print(f"{product:<40} {stats['quantity']:>10,} ${stats['revenue']:>13,.2f} {stats['count']:>8} ${avg_price:>10,.2f}")

## Additional Analysis

### Top Countries by Revenue

In [ ]:
# Top countries using DataFrame API
top_countries = txn_with_total \
    .groupBy("Country") \
    .agg(
        spark_sum("TotalSale").alias("TotalRevenue"),
        spark_sum("Quantity").alias("TotalQuantity"),
        F.count("TransactionNo").alias("NumTransactions")
    ) \
    .orderBy(desc("TotalRevenue")) \
    .limit(10)

print("Top 10 Countries by Revenue:")
top_countries.show(truncate=False)

### Products Never Sold

In [ ]:
# Find products with zero quantity sold
never_sold = txn_df \
    .filter(col("Quantity") == 0) \
    .select("ProductNo", "ProductName") \
    .distinct()

print(f"Products with zero sales: {never_sold.count()}")
never_sold.show(20, truncate=False)

### Save Top Products to File

In [ ]:
# Save top products by revenue to CSV
output_path = "/content/sample_data/top_products_by_revenue"

top_by_revenue.coalesce(1) \
    .write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(output_path)

print(f"Top products saved to: {output_path}")

In [ ]:
# Stop Spark Session
spark.stop()